# Ko-miracl RAG — 생성(Generation) 파이프라인

앞 노트북(`ko_miracl_rag.ipynb`)의 **임베딩 · ChromaDB · retriever** 코드를 재사용하고,
그 위에 **생성형 RAG** 파이프라인을 얹습니다. (LangChain 미사용)

```
사용자의 추상적 질문
   └─▶ ① LLM 질의 재작성 (rewrite)
        └─▶ ② retriever 검색 (ChromaDB, bge-m3)
             └─▶ ③ 프롬프트 생성 (검색 문서 주입)
                  └─▶ ④ LLM 응답 생성
```

- **임베딩(검색)**: `BAAI/bge-m3`
- **LLM(생성)**: `Qwen/Qwen2.5-7B-Instruct` (한국어 지원, Colab T4에서 4-bit 로드)

## 0. 설치 · 설정 (재사용 파트)

In [66]:
!pip -q install -U datasets chromadb sentence-transformers transformers accelerate bitsandbytes

In [67]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ===== 검색 임베딩 모델 (단일: bge-m3) =====
MODEL_CFG = {"model_name": "BAAI/bge-m3", "query_prefix": "", "passage_prefix": ""}

# ===== 생성 LLM (한국어 지원) =====
LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # 메모리 부족 시 'Qwen/Qwen2.5-3B-Instruct' 로 교체

# ===== 하이퍼파라미터 =====
TOP_K = 5
N_EVAL_QUERIES = 30      # 인덱싱할 코퍼스 규모를 줄이려고 작게 설정
NEG_POOL_SIZE  = 3000
BATCH_SIZE = 128
MAX_SEQ_LEN = 512

Device: cuda


In [68]:
from datasets import load_dataset

# queries: 질문 목록 / default(dev): 관련도 라벨 → 데모용 인덱스 구성에 사용
q_dd = load_dataset("taeminlee/Ko-miracl", "queries")
queries = {r["_id"]: r["text"] for r in q_dd[list(q_dd.keys())[0]]}

default_dd = load_dataset("taeminlee/Ko-miracl", "default")
qrels_df = pd.DataFrame(default_dd["dev"])
print("queries:", len(queries), "| qrels(dev):", len(qrels_df))

queries: 2761 | qrels(dev): 3057


### 코퍼스 부분집합 인덱싱 (재사용)
전체 149만 청크 대신 **정답 문서 + 네거티브 풀**만 스트리밍으로 수집합니다.
스트리밍이라 이 셀이 가장 오래 걸립니다 (수 분). 빠르게 보려면 `N_EVAL_QUERIES`를 줄이세요.

In [69]:
qrels_pos = qrels_df[qrels_df["score"] > 0]
eval_qids = qrels_pos["query-id"].unique().tolist()
random.shuffle(eval_qids); eval_qids = eval_qids[:N_EVAL_QUERIES]

gold = {}
for _, row in qrels_df[qrels_df["query-id"].isin(eval_qids)].iterrows():
    gold.setdefault(row["query-id"], {})[row["corpus-id"]] = float(row["score"])
needed_cids = {c for rels in gold.values() for c, s in rels.items() if s > 0}
print("데모 쿼리:", len(eval_qids), "| 정답 문서:", len(needed_cids))

데모 쿼리: 30 | 정답 문서: 100


In [70]:
corpus_dd = load_dataset("taeminlee/Ko-miracl", "corpus", streaming=True)
corpus_stream = corpus_dd[list(corpus_dd.keys())[0]]

docs, found, neg = {}, set(), 0
for row in corpus_stream:
    cid = row["_id"]
    if cid in needed_cids and cid not in docs:
        docs[cid] = {"title": row["title"], "text": row["text"]}; found.add(cid)
    elif neg < NEG_POOL_SIZE:
        docs[cid] = {"title": row["title"], "text": row["text"]}; neg += 1
    if len(found) >= len(needed_cids) and neg >= NEG_POOL_SIZE:
        break
print("수집 청크:", len(docs), "| 정답 확보:", len(found), "/", len(needed_cids))

수집 청크: 3100 | 정답 확보: 100 / 100


### 임베딩 · ChromaDB · retriever (재사용, 단일 모델)

In [71]:
from sentence_transformers import SentenceTransformer
import chromadb

def load_model(cfg):
    m = SentenceTransformer(cfg["model_name"], device=DEVICE); m.max_seq_length = MAX_SEQ_LEN; return m

def embed(model, texts, prefix, batch_size=BATCH_SIZE, show_progress=True):
    texts = [prefix + t for t in texts]
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show_progress)

chroma_client = chromadb.Client()

def build_collection(name, model, cfg, docs):
    try: chroma_client.delete_collection(name)
    except Exception: pass
    col = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    cids = list(docs.keys())
    texts = [docs[c]["text"] for c in cids]; titles = [docs[c]["title"] for c in cids]
    embs = embed(model, texts, cfg["passage_prefix"])
    for i in range(0, len(cids), BATCH_SIZE):
        col.add(ids=cids[i:i+BATCH_SIZE], embeddings=embs[i:i+BATCH_SIZE].tolist(),
                documents=texts[i:i+BATCH_SIZE],
                metadatas=[{"title": t} for t in titles[i:i+BATCH_SIZE]])
    print(f"[{name}] 적재 완료: {col.count()} chunks"); return col

def retrieve(col, model, cfg, query_text, k=TOP_K):
    q_emb = embed(model, [query_text], cfg["query_prefix"], batch_size=1, show_progress=False)
    res = col.query(query_embeddings=q_emb.tolist(), n_results=k)
    out = []
    for cid, doc, dist, meta in zip(res["ids"][0], res["documents"][0],
                                    res["distances"][0], res["metadatas"][0]):
        out.append({"corpus_id": cid, "score": 1 - dist,          # score = 1 - distance
                    "title": meta.get("title", ""), "text": doc})  # 생성용이라 전체 텍스트 유지
    return out

In [72]:
model = load_model(MODEL_CFG)
collection = build_collection("ko-miracl-bge-m3", model, MODEL_CFG, docs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

[ko-miracl-bge-m3] 적재 완료: 3100 chunks


## 1. 생성 LLM 로드 (한국어)
`Qwen2.5-7B-Instruct`를 4-bit로 로드합니다.

In [73]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto")
llm.eval()
print("LLM loaded:", LLM_NAME)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


LLM loaded: Qwen/Qwen2.5-7B-Instruct


In [74]:
@torch.no_grad()
def chat(system, user, max_new_tokens=512, temperature=0.3):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    # 최신 transformers는 return_dict=True로 BatchEncoding을 받아 **enc로 넘기는 게 안전
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=max(temperature, 1e-5),
                       top_p=0.9, pad_token_id=tok.eos_token_id)
    gen = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

## 2. ① 질의 재작성 (Query Rewriting)
사용자의 추상적/모호한 질문을 검색에 적합한 핵심 질의로 재작성합니다.

In [75]:
def rewrite_query(question):
    system = ("당신은 검색 질의 재작성 전문가입니다. 사용자의 모호하거나 추상적인 질문을 "
              "검색엔진에 적합하도록 핵심 개체·키워드 중심의 명확한 한국어 검색 질의로 바꾸세요. "
              "설명 없이 재작성된 질의 한 줄만 출력하세요.")
    user = f"사용자 질문: {question}\n재작성된 검색 질의:"
    return chat(system, user, max_new_tokens=64, temperature=0.0)

## 3. ③ 프롬프트 생성
검색된 문서를 근거로 주입하는 RAG 프롬프트를 만듭니다.

In [76]:
SYSTEM_GEN = ("당신은 주어진 참고 문서를 근거로 한국어로 답하는 assistant입니다. "
              "답변은 한국어로 작성하세요. 고유명사·인명·회사명·전문용어 등은 필요하면 원어를 쓰거나 병기해도 되지만, "
              "문장 자체가 다른 언어로 넘어가면 안 됩니다. "
              "특히 중국어 문장을 출력하지 마세요. 중국어로 표현하고 싶은 내용이 있으면 반드시 한국어로 바꿔 답하세요. "
              "문서에 없는 내용은 지어내지 말고, 근거가 없으면 '제공된 문서에서 찾을 수 없습니다'라고 답하세요. "
              "답변에 사용한 문서 번호를 [문서 n] 형태로 인용하세요.")

def _format_ctx(contexts, snippet_len=800):
    return "\n\n".join(
        f"[문서 {i+1}] (id={c['corpus_id']}, title={c['title']})\n{c['text'][:snippet_len]}"
        for i, c in enumerate(contexts))

# ── A) 베이스라인: 지시만 (기존 프롬프트) ──
def build_prompt(question, contexts, snippet_len=800):
    user = f"# 참고 문서\n{_format_ctx(contexts, snippet_len)}\n\n# 질문\n{question}\n\n# 답변"
    return SYSTEM_GEN, user

# ── B) Few-shot: 다중 인용 답변 1개 + 근거없음→거절 1개를 예시로 주입 ──
_FEWSHOT = """다음은 답변 형식 예시입니다.

# 예시 1 (근거가 된 문서를 모두 인용, 근거 아닌 문서는 인용하지 않음)
# 참고 문서
[문서 1] (id=ex-a, title=에베레스트산)
에베레스트산은 해발 8,848m로 지구에서 가장 높은 산이다.
[문서 2] (id=ex-b, title=백두산)
백두산은 한반도에서 가장 높은 산이다.
[문서 3] (id=ex-c, title=에베레스트산 등반사)
에베레스트산은 네팔과 중국 티베트 자치구의 국경에 걸쳐 있다.
# 질문
세계에서 가장 높은 산은 어디에 있고 높이는 얼마인가요?
# 답변
세계에서 가장 높은 산은 에베레스트산으로, 높이는 해발 8,848m입니다 [문서 1]. 네팔과 중국 티베트 자치구의 국경에 걸쳐 있습니다 [문서 3].

# 예시 2 (문서에 근거 없음 → 지어내지 말고 거절)
# 참고 문서
[문서 1] (id=ex-d, title=커피)
커피는 커피나무 열매의 씨앗을 볶아 만든 음료다.
# 질문
녹차에 들어있는 카페인 함량은?
# 답변
제공된 문서에서 찾을 수 없습니다.

이제 아래 실제 질문에 위 형식으로 답하세요.
"""

def build_prompt_fewshot(question, contexts, snippet_len=800):
    user = (_FEWSHOT
            + f"\n# 참고 문서\n{_format_ctx(contexts, snippet_len)}"
            + f"\n\n# 질문\n{question}\n\n# 답변")
    return SYSTEM_GEN, user

## 4. 전체 파이프라인
사용자 질문 → ① 재작성 → ② 검색 → ③ 프롬프트 → ④ 응답

In [77]:
def rag_answer(question, k=TOP_K, verbose=True, prompt_fn=build_prompt):
    # ① 질의 재작성
    rq = rewrite_query(question)
    # ② retriever 검색
    ctxs = retrieve(collection, model, MODEL_CFG, rq, k=k)
    # ③ 프롬프트 생성 (prompt_fn 으로 A/B 교체)
    system, user = prompt_fn(question, ctxs)
    # ④ LLM 응답 생성
    answer = chat(system, user, max_new_tokens=512, temperature=0.2)

    if verbose:
        print("① 원 질문     :", question)
        print("① 재작성 질의 :", rq)
        print("② 검색 결과 (score = 1 - distance):")
        for i, c in enumerate(ctxs, 1):
            print(f"   [{i}] score={c['score']:.4f} id={c['corpus_id']} | {c['title']}")
        print("\n④ 생성 응답 :\n" + answer)
    return {"question": question, "rewritten": rq, "contexts": ctxs, "answer": answer}

## 5. 실행 예시

In [78]:
# 예시 A: Ko-miracl dev의 실제 질문 (정답 문서가 인덱스에 포함되어 있음)
_ = rag_answer(queries[eval_qids[0]])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


① 원 질문     : 구지가는 고대 시가의 하나인가요?
① 재작성 질의 : 구지가가 고대 시인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6918 id=264838#12 | 한국의 상고시대 시가
   [2] score=0.4292 id=986864#0 | 양해 (후한)
   [3] score=0.4098 id=987124#0 | 요시카와 코지로
   [4] score=0.3878 id=989604#0 | 트라시마코스
   [5] score=0.3871 id=988540#71 | 왕자 조

④ 생성 응답 :
네, 구지가는 고대 시가의 하나입니다. [문서 1]에 따르면, 구지가는 서기 40년경에 이루어진 고대 시가 중 하나입니다.


In [79]:
_ = rag_answer(queries[eval_qids[12]])

① 원 질문     : 한국어는 언제 만들어졌나요?
① 재작성 질의 : 한국어가 언제 시작되었나요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6093 id=487516#0 | 한글
   [2] score=0.4787 id=1852#9 | 유럽
   [3] score=0.4708 id=261019#0 | 불교의 역사
   [4] score=0.4509 id=987704#3 | 학생신문
   [5] score=0.4459 id=3966#1 | 한국의 성씨와 이름

④ 생성 응답 :
한국어는 1446년에 훈민정음이라는 이름으로 창제되었습니다. [문서 1]에 따르면, 조선 제 4대 임금 세종이 훈민정음을 창제하고 1446년에 반포하였습니다.


In [80]:
# 예시 B: 사용자가 던지는 추상적/모호한 질문
_ = rag_answer("그 유명한 상대성 이론 만든 사람 어떤 인물이야?")

① 원 질문     : 그 유명한 상대성 이론 만든 사람 어떤 인물이야?
① 재작성 질의 : 상대성 이론을 제안한 사람 누구인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.4527 id=3533#0 | 르네 데카르트
   [2] score=0.4490 id=985079#3 | 마이어-피토리스 열
   [3] score=0.4317 id=780040#17 | 라이트 판타스틱
   [4] score=0.4213 id=987149#3 | 제임스 워델 알렉산더
   [5] score=0.4148 id=987221#0 | 류드비크 파데예프

④ 생성 응답 :
제공된 참고 문서에서 상대성 이론을 만든 사람에 대해 언급되어 있지 않습니다. 상대성 이론은 아인슈타인에 의해 개발되었으나, 현재 제공된 문서들은 르네 데카르트, 발터 마이어, 레오폴트 피토리스, 제임스 워델 알렉산더, 류드비크 파데예프 등과 관련된 내용만 다루고 있습니다. 따라서 이 질문에 대한 답변을 제공하기 위해서는 다른 참고 자료가 필요합니다.


In [81]:
_ = rag_answer("물 몇도에 끓음?")

① 원 질문     : 물 몇도에 끓음?
① 재작성 질의 : 물이 끓는 온도는 몇 도인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.5234 id=986668#2 | 2003년 유럽 폭염
   [2] score=0.5144 id=985918#0 | 로스팅
   [3] score=0.4879 id=987625#13 | 주위
   [4] score=0.4788 id=47536#1 | 우라늄
   [5] score=0.4788 id=986668#1 | 2003년 유럽 폭염

④ 생성 응답 :
제공된 문서에서 찾을 수 없습니다.


In [82]:
_ = rag_answer("키보드 주문제작 하고 싶어")

① 원 질문     : 키보드 주문제작 하고 싶어
① 재작성 질의 : 키보드 제작 서비스 찾기
② 검색 결과 (score = 1 - distance):
   [1] score=0.6641 id=985068#3 | 커스텀 키보드
   [2] score=0.6520 id=985068#4 | 커스텀 키보드
   [3] score=0.6363 id=985068#2 | 커스텀 키보드
   [4] score=0.6326 id=985068#0 | 커스텀 키보드
   [5] score=0.6277 id=985068#1 | 커스텀 키보드

④ 생성 응답 :
키보드 주문제작을 원하신다면 두 가지 방법이 있습니다. 하나는 직접 설계와 제작을 하되, 이는 전문 지식이 필요하고 비용이 많이 들 수 있으므로 몇몇 사용자들을 제외하고는 현실적으로 어려울 수 있습니다. 다른 하나는 공동으로 제작하는 방법으로, 전문 지식이 필요하지 않으며 비교적 저렴하게 커스텀 키보드를 제작할 수 있습니다. 주문제작을 원하신다면 공동 제작을 고려해보시는 것이 좋을 것 같습니다. [문서 1], [문서 2]


## 6. 프롬프트 A/B 비교 (지표 기반 채택)
같은 검색 결과 위에서 **A(지시형) vs B(few-shot)** 응답을 생성해 비교합니다.
검색은 프롬프트와 무관하므로 쿼리당 한 번만 수행해 공유하고, 생성 프롬프트만 교체합니다.

| 지표 | 의미 | 방향 |
|---|---|---|
| **인용recall(정답검색시)** | 정답 문서가 검색된 쿼리에서, 답변이 그 정답 문서를 `[문서 n]`으로 인용한 비율 | 높을수록 좋음 |
| **거절율(정답없을때)** | 정답 문서가 검색되지 않은 쿼리에서, 지어내지 않고 거절한 비율 | 높을수록 좋음 |
| **포맷준수율** | 인용 또는 거절 형식을 지킨 비율 | 높을수록 좋음 |

세 지표에서 앞서는 프롬프트를 `rag_answer(..., prompt_fn=...)` 기본값으로 채택하면 됩니다.

In [83]:
# Google Drive 마운트 + 검색 캐시 경로
# 프롬프트 실험은 검색 결과가 불변 → 한 번 검색해 Drive에 저장, 재시작 후엔 로드만
from google.colab import drive
drive.mount("/content/drive")

import os, glob, json
# 내 드라이브 > MS_AI_NLP(2026)_... > 22_실전프로젝트2
DRIVE_DIR = glob.glob("/content/drive/MyDrive/*/22_실전프로젝트2")[0]
CACHE_DIR = os.path.join(DRIVE_DIR, "rag_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
SEARCH_CACHE = os.path.join(CACHE_DIR, "searched.json")
print("검색 캐시:", SEARCH_CACHE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
검색 캐시: /content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/22_실전프로젝트2/rag_cache/searched.json


In [ ]:
import re
from tqdm.auto import tqdm

def _cited_ids(answer, ctxs):
    # 답변의 인용 → 실제 corpus_id 집합
    # [문서 1] 뿐 아니라 [문서 1, 문서 3, 문서 4], [문서 1, 3] 같은 묶음 인용도 파싱
    idxs = {int(n) for grp in re.findall(r"\[문서[^\]]*\]", answer)
                   for n in re.findall(r"\d+", grp)}
    return {ctxs[i-1]["corpus_id"] for i in idxs if 1 <= i <= len(ctxs)}

def search_all(qids, k=TOP_K):
    """① 검색만 수행. 결과는 JSON 직렬화 가능 → Drive 캐시 후 세션 재시작 시 재사용"""
    searched = []
    for qid in tqdm(qids, desc="① 검색"):
        q = queries[qid]
        ctxs = retrieve(collection, model, MODEL_CFG, rewrite_query(q), k=k)
        gold_cids = {cid for cid, s in gold.get(qid, {}).items() if s > 0}
        gold_hit = gold_cids & {c["corpus_id"] for c in ctxs}
        searched.append({"qid": qid, "q": q, "ctxs": ctxs, "gold_hit": sorted(gold_hit)})
    n_hit = sum(bool(s["gold_hit"]) for s in searched)
    print(f"평가셋 구성: 정답검색됨 {n_hit} / 미검색(거절 기대) {len(searched) - n_hit}")
    # 미검색이 0이면 '거절율' 비교가 불가능 → 전체 실행 전에 여기서 판단
    return searched

def eval_prompts(prompt_fns, searched, max_new_tokens=256):
    """② 같은 검색 결과(searched) 위에서 프롬프트만 바꿔 생성 → 지표 비교"""
    rows = {name: [] for name in prompt_fns}
    bar = tqdm(searched, desc="② 생성")
    for s in bar:
        gold_hit = set(s["gold_hit"])
        for name, fn in prompt_fns.items():
            bar.set_postfix_str(f"{name} 생성중")
            system, user = fn(s["q"], s["ctxs"])
            ans = chat(system, user, max_new_tokens=max_new_tokens, temperature=0.2)
            cited = _cited_ids(ans, s["ctxs"])
            abstained = "찾을 수 없습니다" in ans
            rows[name].append({
                "qid": s["qid"],
                "answerable": bool(gold_hit),
                "cite_gold": bool(cited & gold_hit),      # 정답 문서를 인용했나
                "abstained": abstained,
                "fmt_ok": bool(cited) or abstained,        # 인용 or 거절 형식 준수
                "ans": ans,                                # 실패 케이스 원문 확인용
            })
    return rows

def summarize(rows):
    out = []
    for name, rs in rows.items():
        ans_q = [r for r in rs if r["answerable"]]      # 정답이 검색된 쿼리
        no_q  = [r for r in rs if not r["answerable"]]  # 정답이 안 검색된 쿼리
        out.append({
            "prompt": name,
            "인용recall(정답검색시)": round(np.mean([r["cite_gold"] for r in ans_q]), 3) if ans_q else None,
            "거절율(정답없을때)":     round(np.mean([r["abstained"] for r in no_q]), 3) if no_q else None,
            "포맷준수율":             round(np.mean([r["fmt_ok"] for r in rs]), 3),
            "정답검색/미검색":        f"{len(ans_q)}/{len(no_q)}",
            "n": len(rs),
        })
    return pd.DataFrame(out)

In [85]:
# ── 첫 실행: 검색 수행 → Drive에 캐시 저장 ──
# 먼저 5개로 파이프라인·ETA 확인 → 괜찮으면 eval_qids 로 바꿔 전체 실행
searched = search_all(eval_qids[:5])
json.dump(searched, open(SEARCH_CACHE, "w"), ensure_ascii=False)
print("캐시 저장 완료:", SEARCH_CACHE)

# ── 세션 재시작 시: 위 세 줄 대신 아래 한 줄만 ──
# (0.설치 → 1.LLM 로드 → 3.프롬프트 → 마운트 셀 → 정의 셀만 실행하면 됨.
#  코퍼스 수집·임베딩·ChromaDB·bge-m3 전부 불필요)
# searched = json.load(open(SEARCH_CACHE))

① 검색:   0%|          | 0/5 [00:00<?, ?it/s]

평가셋 구성: 정답검색됨 5 / 미검색(거절 기대) 0
캐시 저장 완료: /content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/22_실전프로젝트2/rag_cache/searched.json


In [86]:
# ② 생성 A/B (검색 결과는 searched 공유)
rows = eval_prompts({"A_baseline": build_prompt, "B_fewshot": build_prompt_fewshot}, searched)
summarize(rows)

② 생성:   0%|          | 0/5 [00:00<?, ?it/s]

,prompt,인용recall(정답검색시),거절율(정답없을때),포맷준수율,정답검색/미검색,n
0,A_baseline,0.8,None,1.0,5/0,5
1,B_fewshot,0.6,None,0.8,5/0,5


In [87]:
# 실패 케이스 원문 확인: 정답이 검색됐는데 인용하지 못한 답변
def show_failures(rows, name):
    fails = [r for r in rows[name] if r["answerable"] and not r["cite_gold"]]
    print(f"[{name}] 인용 실패 {len(fails)}건")
    for r in fails:
        print(f"\n── qid={r['qid']} | Q: {queries[r['qid']]}")
        print(r["ans"])

show_failures(rows, "B_fewshot")

[B_fewshot] 인용 실패 2건

── qid=405 | Q: 필리핀의 수도는 어디인가요?
필리핀의 수도는 메트로 마닐라로, 그 중 수도로서의 역할을 하는 도시는 메트로 마닐라의 일부인 메트로 마닐라 특별시인 마닐라이다 [문서 1, 문서 3, 문서 4].

── qid=1011 | Q: 힌드교 창시자는 누구인가요?
제공된 문서에서 찾을 수 없습니다. 문서들은 불교와 관련된 내용을 다루고 있지만, 힌드교 창시자에 대한 정보는 포함되어 있지 않습니다.


### 더 해볼 것
- 재작성을 **다중 질의(multi-query)** 로 확장해 검색 recall 향상
- 리랭커(`BAAI/bge-reranker-v2-m3`)로 검색 결과 재정렬 후 상위 문서만 프롬프트에 주입
- 답변의 인용 문서와 정답셋(`gold`)을 비교해 groundedness 평가
- Qwen vs GPT 비교